### functional

In [ ]:
def objective(params):
    import pandas as pd
    from sklearn.datasets import load_iris

    iris = load_iris()
    data = pd.DataFrame(iris["data"], columns=iris["feature_names"])
    target = pd.DataFrame(iris["target"], columns=["target"])

    from sklearn.model_selection import train_test_split
    X_train, X_val, y_train, y_val = train_test_split(data.values, target.values, test_size=0.2)

    import logging

    logging.basicConfig(
        format="%(asctime)s %(levelname)-8s %(message)s",
        datefmt="%Y-%m-%dT%H:%M:%SZ",
        level=logging.INFO,
    )    

    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score
    clf = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("rf", RandomForestClassifier(
                    n_estimators=int(params['n_estimators']),
                    max_depth=int(params['max_depth']),
                    criterion = params['criterion'],
                    oob_score=True
                ))
        ]
    )
    clf.fit(X_train, y_train.ravel())
    y_pred = clf.predict(X_val)
    logging.info(f"accuracy={accuracy_score(y_val, y_pred)}")

In [ ]:
from kubeflow import katib

params = {
    "n_estimators": katib.search.int(min=100, max=1000, step=100),
    "max_depth": katib.search.int(min=2, max=6, step=1),
    "criterion": katib.search.categorical(['gini', 'entropy', 'log_loss'])
}

EXP_NAME = "iris"
NAMESPANCE = 'kubeflow-user-example-com'
katib_client = katib.KatibClient()

katib_client.tune(
    name=EXP_NAME,
    namespace=NAMESPANCE,
    objective=objective, 
    parameters=params, 
    algorithm_name="bayesianoptimization", 
    objective_metric_name="accuracy", 
    objective_type= 'maximize',
    # additional_metric_names=["loss"], 
    max_trial_count=20, 
    base_image='python:3.8.10-slim',
    packages_to_install=['scikit-learn', 'pandas']
)

In [ ]:
katib_client.create_experiment()

In [ ]:
status = katib_client.is_experiment_succeeded(EXP_NAME, namespace=NAMESPANCE)
print(f"Katib Experiment is Succeeded: {status}\n")

In [ ]:
best_hps = katib_client.get_optimal_hyperparameters(EXP_NAME, namespace=NAMESPANCE)

In [ ]:
katib_client.get_suggestion(EXP_NAME, namespace=NAMESPANCE)

### katib pipeline

In [1]:
from functools import partial
from typing import NamedTuple

from kfp.components import InputPath, OutputPath
from kfp.components import create_component_from_func

@partial(
    create_component_from_func,
    base_image="python:3.8.10-slim",
    packages_to_install=["pandas", "scikit-learn"]
)
def load_iris_data(
    data_path: OutputPath("csv"),
    target_path: OutputPath("csv"),
):
    import pandas as pd
    from sklearn.datasets import load_iris
    iris = load_iris()
    data = pd.DataFrame(iris["data"], columns=iris["feature_names"])
    target = pd.DataFrame(iris["target"], columns=["target"])

    data.to_csv(data_path, index=False)
    target.to_csv(target_path, index=False)


In [3]:
import kfp
import kfp.dsl as dsl
from kfp import components

from kubeflow.katib import ApiClient
from kubeflow.katib import V1beta1ExperimentSpec
from kubeflow.katib import V1beta1AlgorithmSpec
from kubeflow.katib import V1beta1EarlyStoppingSpec
from kubeflow.katib import V1beta1EarlyStoppingSetting
from kubeflow.katib import V1beta1ObjectiveSpec
from kubeflow.katib import V1beta1ParameterSpec
from kubeflow.katib import V1beta1FeasibleSpace
from kubeflow.katib import V1beta1TrialTemplate
from kubeflow.katib import V1beta1TrialParameterSpec
from kubeflow.katib import V1beta1MetricsCollectorSpec
from kubeflow.katib import V1beta1MetricStrategy

objective=V1beta1ObjectiveSpec(
    type="maximize",
    # metric_strategies= V1beta1MetricStrategy(name='Validation-accuracy', value='max')
    goal= 0.99,
    objective_metric_name="accuracy"
)

algorithm=V1beta1AlgorithmSpec(
    algorithm_name="bayesianoptimization",
)

early_stopping=V1beta1EarlyStoppingSpec(
    algorithm_name="medianstop",
    algorithm_settings=[
        V1beta1EarlyStoppingSetting(
            name="min_trials_required",
            value="2"
        )
    ]
)

parameters=[
    V1beta1ParameterSpec(
        name="n_estimators",
        parameter_type="int",
        feasible_space=V1beta1FeasibleSpace(
            min="100",
            max="1000",
            step="100"
        ),
    ),
    V1beta1ParameterSpec(
        name="max_depth",
        parameter_type="int",
        feasible_space=V1beta1FeasibleSpace(
            min="2",
            max="6",
            step="1"
        ),
    ),
    V1beta1ParameterSpec(
        name="criterion",
        parameter_type="categorical",
        feasible_space=V1beta1FeasibleSpace(
            list=[
                "gini", 
                "entropy",
                "log_loss"
            ]
        ),
    ),
]

metric_collector = V1beta1MetricsCollectorSpec(collector='StdOut')

# JSON template specification for the Trial's Worker Kubernetes Job.
trial_spec={
    "apiVersion": "batch/v1",
    "kind": "Job",
    "spec": {
        "template": {
            "metadata": {
                "annotations": {
                     "sidecar.istio.io/inject": "false"
                }
            },
            "spec": {
                "containers": [
                    {
                        "name": "training-container",
                        "image": "192.168.0.50:5100/iris-trainer:0.1",
                        "command": [
                            "python",
                            "trainer.py",
                            "--criterion=${trialParameters.criterion}",
                            "--n_estimators=${trialParameters.nEstimators}",
                            "--max_depth=${trialParameters.maxDepth}",
                            "--train_ds_path=/mnt/train-ds.csv"
                        ],
                        "volumeMounts": [
                            {
                                "mountPath": "/mnt",
                                "name": "katib-mnt"
                            }
                        ]
                    }
                ],
                "volumes": [
                    {
                        "name": "katib-mnt",
                        "persistentVolumeClaim": {
                            "claimName": "pvc-iris"
                        }
                    }
                ],
                "restartPolicy": "Never",
            }
        }
    }
}

In [ ]:
trial_template=V1beta1TrialTemplate(
    retain=True,
    primary_container_name="training-container",
    trial_parameters=[
        V1beta1TrialParameterSpec(
            name="criterion",
            description="criterion for the training model",
            reference="criterion"
        ),
        V1beta1TrialParameterSpec(
            name="nEstimators",
            description="Number of training model estimators",
            reference="n_estimators"
        ),
        V1beta1TrialParameterSpec(
            name="maxDepth",
            description="Training model maxDepth",
            reference="max_depth"
        ),
    ],
    trial_spec=trial_spec
)

In [ ]:
max_trial_count = 10
max_failed_trial_count = 3
parallel_trial_count = 2

experiment_spec=V1beta1ExperimentSpec(
    max_trial_count=max_trial_count,
    max_failed_trial_count=max_failed_trial_count,
    parallel_trial_count=parallel_trial_count,
    objective=objective,
    algorithm=algorithm,
    early_stopping=early_stopping,
    parameters=parameters,
    trial_template=trial_template,
    metrics_collector_spec=metric_collector,
)

katib_experiment_launcher_op = components.load_component_from_url(
    "https://raw.githubusercontent.com/kubeflow/pipelines/master/components/kubeflow/katib-launcher/component.yaml")

In [ ]:
def convert_katib_results(katib_results):
    import json
    katib_res = json.loads(katib_results)

    convert_res = {}
    for p in katib_res['currentOptimalTrial']["parameterAssignments"]:
        convert_res[p['name']] = p['value']

    with open('mnt/hyps.json', 'w') as f:
        json.dump(convert_res, f)

In [ ]:
from kfp import onprem

@dsl.pipeline(
    name="Launch Katib Experiment",
    description="An example to launch Katib Experiment with early stopping"
)
def penguins_hyps():
    # Katib launcher component.
    # Experiment Spec should be serialized to a valid Kubernetes object.
    # Output container to print the results.
    katib_op = katib_experiment_launcher_op(
        experiment_name=EXP_NAME,
        experiment_namespace="kubeflow-user-example-com",
        experiment_spec=ApiClient().sanitize_for_serialization(experiment_spec),
        experiment_timeout_minutes=3,
        delete_finished_experiment=False)
    
    convert_katib_results_op = components.func_to_container_op(convert_katib_results)
    best_hp_op = convert_katib_results_op(katib_op.output)
    best_hp_op.apply(onprem.mount_pvc(pvc_name='pvc-penguins', volume_name= 'data-mnt',volume_mount_path='/mnt'))
        
    op_out = dsl.ContainerOp(
        name="best-hp",
        image="library/bash:4.4.23",
        command=["sh", "-c"],
        arguments=["echo Best HyperParameters: %s" % katib_op.output],
    )